# Tome 14, Chapter 1: Additive functions and the Kubilius model (Python companion)

Exercises first, solutions at the end. We sieve $\varphi(n)/n$ and $\sigma(n)/n$ for all $n\le10^6$, compare the law of the small-prime part of $\omega(n)$ with the independent Kubilius model in total variation, simulate the Erdős–Wintner limit law, and test the Littlewood–Offord bound. The computer science application: **sieving multiplicative functions in $O(N\log\log N)$** instead of factoring each $n$. Requires `numpy`.

In [ ]:
import math, time
import numpy as np

def primes_upto(N):
    s = np.ones(N + 1, dtype=bool); s[:2] = False
    for i in range(2, int(N ** 0.5) + 1):
        if s[i]: s[i * i::i] = False
    return np.nonzero(s)[0]

N = 10**6
P = primes_upto(N)

## Exercises

**P1.1 (★) Sieving $\varphi(n)/n$.** Compute $\varphi(n)/n$ for all $n\le10^6$ by a sieve over the primes, check $\frac1N\sum_{n\le N}\varphi(n)/n$ against $6/\pi^2$ (Exercise 1.1), and plot the empirical distribution function (Figure 1.1).

**P1.2 (★) Abundant numbers.** Compute $\sigma(n)/n=\sum_{d\mid n}1/d$ for all $n\le10^6$ with a divisor sieve, count the abundant numbers up to $10^4,10^5,10^6$, and compare with the limit $0.24761\ldots$ Which is the first odd abundant number?

**P1.3 (★★) The Kubilius model in total variation.** For $x=10^6$ and $y\in\{3,5,7,11,13\}$, compute the exact law of $\omega_y(n)=\#\{p\le y: p\mid n\}$ for $n\le x$ and of the model $\sum_{p\le y}B_p$ with independent Bernoulli$(1/p)$, and their total variation distance. Then do the same for $y=31,101$. How does the distance compare with $e^{-u/4}$, $u=\log x/\log y$?

**P1.4 (★★) The Erdős–Wintner limit law by simulation.** Simulate $Z=\sum_{p\le 10^5}\log(1-1/p)\,\mathbf 1_{G_p\ge1}$ ($2\cdot10^4$ samples) and compare its distribution function, through $e^Z$, with the empirical one of $\varphi(n)/n$. Compute the Kolmogorov distance.

**P1.5 (★★) Littlewood–Offord.** For $m=20$ and (a) $a_i=1$, (b) $a_i=i$, (c) $a_i$ random reals, compute $\max_c P(\sum\varepsilon_ia_i=c)$ exactly by enumerating the $2^{20}$ sign vectors (or by a dictionary convolution), and compare with $\binom{m}{m/2}2^{-m}$ and $1/\sqrt m$.

**P1.6 (★★) Computer science: sieving versus factoring.** Time three ways of computing $\sigma(n)$ for all $n\le N$: (i) trial division of each $n$, (ii) the divisor sieve of P1.2 (cost $\sum_{d\le N}N/d\approx N\log N$), (iii) a linear sieve using the smallest prime factor and multiplicativity. Compare for $N=10^4$ and $10^5$, and estimate the cost for $10^6$.

## Solutions

### P1.1

In [ ]:
phi = np.arange(N + 1, dtype=float)
for p in P: phi[p::p] *= (1 - 1 / p)
r = phi[1:] / np.arange(1, N + 1)
print("mean of phi(n)/n:", r.mean(), " 6/pi^2 =", 6 / math.pi**2)
xs = np.sort(r)
for z in [0.25, 1/3, 0.4, 0.5, 0.6, 0.8]:
    print(f"P(phi(n)/n <= {z:.3f}) = {np.searchsorted(xs, z, side='right') / N:.4f}")

The jumps at $z=1/2$ and $1/3$ in the empirical function are steep climbs, not jumps: Corollary 1.9 says the limit has no atom. Plotting (with matplotlib) `plt.plot(xs, np.arange(1,N+1)/N)` reproduces Figure 1.1.

### P1.2

In [ ]:
sig = np.zeros(N + 1)
for d in range(1, N + 1): sig[d::d] += 1.0 / d
for X in [10**4, 10**5, 10**6]:
    print(X, (sig[1:X + 1] >= 2 - 1e-12).mean())
odd = next(n for n in range(1, N + 1, 2) if sig[n] >= 2 - 1e-12)
print("first odd abundant number:", odd, " sigma(n)/n =", sig[odd])

The proportions drift down to $0.2476$ slowly: $0.2492$, $0.2480$, $0.2475$; the last one is already below the limit $0.24762$, so the convergence is not monotone. The first odd abundant number is $945=3^3\cdot5\cdot7$: odd numbers need many small odd primes to reach $\sigma(n)\ge2n$.

### P1.3

In [ ]:
def tv_omega_y(x, y):
    ps = [int(p) for p in P if p <= y]
    w = np.zeros(x + 1, dtype=np.int64)
    for p in ps: w[p::p] += 1
    emp = np.bincount(w[1:], minlength=len(ps) + 1) / x
    mod = np.array([1.0])
    for p in ps: mod = np.convolve(mod, [1 - 1 / p, 1 / p])
    k = max(len(emp), len(mod))
    emp = np.pad(emp, (0, k - len(emp))); mod = np.pad(mod, (0, k - len(mod)))
    return 0.5 * np.abs(emp - mod).sum()

x = 10**6
for y in [3, 5, 7, 11, 13, 31, 101, 1009]:
    u = math.log(x) / math.log(y)
    print(f"y={y:5d}  u={u:5.2f}  TV={tv_omega_y(x, y):.2e}  e^(-u/4)={math.exp(-u / 4):.2e}")

For small $y$ the distance is tiny, of the order of $\prod_{p\le y}p/x$ (the residue system modulo $\prod p$ fits many times in $[1,x]$). It grows with $y$, but remains far below the bound $e^{-u/4}$, which is a worst case over all events; the true decay is like $u^{-u}$.

### P1.4

In [ ]:
rng = np.random.default_rng(1)
Pz = P[P <= 10**5].astype(float)
lw = np.log(1 - 1 / Pz)
S = 20000
Z = np.zeros(S)
for i in range(0, len(Pz), 500):
    B = rng.random((S, len(Pz[i:i+500]))) < 1 / Pz[i:i+500]
    Z += B @ lw[i:i+500]
model = np.sort(np.exp(Z))
grid = np.linspace(0, 1, 2001)
Fe = np.searchsorted(xs, grid, side='right') / N
Fm = np.searchsorted(model, grid, side='right') / S
print("Kolmogorov distance:", np.abs(Fe - Fm).max())

The distance is about $0.01$, the size of the Monte Carlo error $1/\sqrt{S}$ plus the effect of truncating at $10^5$ and of $x=10^6$ being finite.

### P1.5

In [ ]:
from collections import Counter
def max_atom(a):
    dist = Counter({0.0: 1})
    for ai in a:
        new = Counter()
        for s, c in dist.items():
            new[round(s + ai, 9)] += c; new[round(s - ai, 9)] += c
        dist = new
    return max(dist.values()) / 2 ** len(a)
m = 20
print("a_i = 1     :", max_atom([1] * m))
print("a_i = i     :", max_atom(list(range(1, m + 1))))
print("a_i random  :", max_atom(list(np.random.default_rng(0).random(m))))
print("binomial bound:", math.comb(m, m // 2) / 2**m, " 1/sqrt(m):", 1 / math.sqrt(m))

Equal coefficients attain the bound; distinct integers give about $m^{-3/2}$ (Sárközy–Szemerédi); generic reals give $2^{-m}$: every sign vector gives a different sum.

### P1.6

In [ ]:
def sigma_trial(N):
    out = [0] * (N + 1)
    for n in range(1, N + 1):
        s, d = 0, 1
        while d * d <= n:
            if n % d == 0:
                s += d + (n // d if d * d != n else 0)
            d += 1
        out[n] = s
    return out

def sigma_divisor_sieve(N):
    s = np.zeros(N + 1, dtype=np.int64)
    for d in range(1, N + 1): s[d::d] += d
    return s

def sigma_linear(N):
    spf = np.zeros(N + 1, dtype=np.int64)
    for p in primes_upto(N)[::-1]: spf[p::p] = p
    s = np.ones(N + 1, dtype=np.int64); pw = np.ones(N + 1, dtype=np.int64)  # pw[n] = p^k || n for p = spf(n)
    s[0] = 0
    for n in range(2, N + 1):
        p = spf[n]; m = n // p
        if m % p == 0:
            pw[n] = pw[m] * p
            q = n // pw[n]
            s[n] = s[q] * (pw[n] * p - 1) // (p - 1)
        else:
            pw[n] = p; s[n] = s[m] * (p + 1)
    return s

for Nn in [10**4, 10**5]:
    t = time.time(); a = sigma_trial(Nn); t1 = time.time() - t
    t = time.time(); b = sigma_divisor_sieve(Nn); t2 = time.time() - t
    t = time.time(); c = sigma_linear(Nn); t3 = time.time() - t
    assert list(b) == a and list(c) == a
    print(f"N={Nn}: trial {t1:.2f}s, divisor sieve {t2:.3f}s, linear sieve {t3:.2f}s")

Trial division costs about $N^{3/2}$ operations and its time grows by a factor $30$ from $10^4$ to $10^5$; the two sieves grow linearly (up to a logarithm). The divisor sieve costs $\sum_{d\le N}N/d\approx N\log N$ additions, the linear sieve one multiplication per $n$; for $N=10^6$ trial division would take about a minute, the sieves about a second.